In [1]:
# ==============================================================================
# ΜΗΝΑΣ 1: ΟΛΟΚΛΗΡΩΜΕΝΟ DATA COLLECTION PIPELINE (10 COINS x 3 SOURCES)
# ==============================================================================

import pandas as pd
import numpy as np
import yfinance as yf
import requests
import time
import warnings
warnings.filterwarnings('ignore')

# Ορίζουμε τα 10 νομίσματα του Γράφου μας
coins_yahoo = ['BTC-USD', 'ETH-USD', 'XRP-USD', 'LTC-USD', 'ADA-USD', 'BNB-USD', 'SOL-USD', 'DOGE-USD', 'TRX-USD', 'LINK-USD']

# Τα αντίστοιχα σύμβολα στα ανταλλακτήρια
coins_binance = ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'LTCUSDT', 'ADAUSDT', 'BNBUSDT', 'SOLUSDT', 'DOGEUSDT', 'TRXUSDT', 'LINKUSDT']
coins_coinbase = ['BTC-USD', 'ETH-USD', 'XRP-USD', 'LTC-USD', 'ADA-USD', 'BNB-USD', 'SOL-USD', 'DOGE-USD', 'TRX-USD', 'LINK-USD']

print("Οι βιβλιοθήκες φορτώθηκαν. Έτοιμοι για μαζική συλλογή δεδομένων!")

Οι βιβλιοθήκες φορτώθηκαν. Έτοιμοι για μαζική συλλογή δεδομένων!


In [2]:
# ==============================================================================
# ΣΥΝΑΡΤΗΣΕΙΣ ΑΝΤΛΗΣΗΣ ΔΕΔΟΜΕΝΩΝ ΑΝΑ ΠΗΓΗ
# ==============================================================================

def get_yahoo_10_coins(start_date='2018-01-01'):
    print("-> Άντληση δεδομένων από Yahoo Finance...")
    df_list = []
    for sym in coins_yahoo:
        df = yf.download(sym, start=start_date, progress=False)
        if not df.empty:
            df = df[['Close']].copy()
            df.columns = [sym] # Ονομάζουμε τη στήλη με το όνομα του νομίσματος
            df_list.append(df)
    
    final_df = pd.concat(df_list, axis=1)
    final_df.index = pd.to_datetime(final_df.index).normalize()
    return final_df

def get_binance_10_coins(limit=1000):
    print("-> Άντληση δεδομένων από Binance API...")
    df_list = []
    for i, sym in enumerate(coins_binance):
        try:
            url = "https://api.binance.com/api/v3/klines"
            params = {'symbol': sym, 'interval': '1d', 'limit': limit}
            res = requests.get(url, params=params).json()
            
            if isinstance(res, list) and len(res) > 0:
                df = pd.DataFrame(res, columns=['Open Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close Time', 'QAV', 'NoT', 'TBB', 'TBQ', 'Ignore'])
                df['Date'] = pd.to_datetime(df['Open Time'], unit='ms').dt.normalize()
                df.set_index('Date', inplace=True)
                df = df[['Close']].astype(float)
                
                # Μετονομασία σε κοινό format (Yahoo format) για να ταιριάζουν τα μοντέλα
                df.columns = [coins_yahoo[i]]
                df_list.append(df)
            time.sleep(0.5)
        except Exception as e:
            print(f"   [!] Σφάλμα στο {sym} (Binance): {e}")
            
    # Μέσα στην get_coinbase_10_coins, στο τέλος της
    final_df = pd.concat(df_list, axis=1).sort_index()
    final_df = final_df.reindex(columns=coins_yahoo)
    return final_df

def get_coinbase_10_coins():
    print("-> Άντληση δεδομένων από Coinbase API...")
    df_list = []
    for i, sym in enumerate(coins_coinbase):
        try:
            url = f"https://api.exchange.coinbase.com/products/{sym}/candles"
            params = {'granularity': 86400} # Ημερήσια (86400 δευτερόλεπτα)
            headers = {"Accept": "application/json"}
            res = requests.get(url, headers=headers, params=params).json()
            
            if isinstance(res, list) and len(res) > 0:
                df = pd.DataFrame(res, columns=['Time', 'Low', 'High', 'Open', 'Close', 'Volume'])
                df['Date'] = pd.to_datetime(df['Time'], unit='s').dt.normalize()
                df.set_index('Date', inplace=True)
                df = df[['Close']].astype(float)
                
                df.columns = [coins_yahoo[i]]
                df_list.append(df)
            else:
                print(f"   [i] Το νόμισμα {sym} δεν βρέθηκε στο Coinbase (Θα μπει NaN).")
            time.sleep(0.5)
        except Exception as e:
            print(f"   [!] Σφάλμα στο {sym} (Coinbase): {e}")
            
    # Μέσα στην get_coinbase_10_coins, στο τέλος της
    final_df = pd.concat(df_list, axis=1).sort_index()
    final_df = final_df.reindex(columns=coins_yahoo)
    return final_df

In [3]:
# ==============================================================================
# ΕΚΤΕΛΕΣΗ, LOG-RETURNS & ΕΞΑΓΩΓΗ ΣΕ CSV
# ==============================================================================

# 1. Κατέβασμα των τιμών κλεισίματος (Close Prices)
prices_yahoo = get_yahoo_10_coins()
prices_binance = get_binance_10_coins()
prices_coinbase = get_coinbase_10_coins()

print("\n--- Σύνοψη Διαστάσεων (Close Prices) ---")
print(f"Yahoo Finance : {prices_yahoo.shape} (Ημέρες, Νομίσματα)")
print(f"Binance       : {prices_binance.shape} (Ημέρες, Νομίσματα)")
print(f"Coinbase      : {prices_coinbase.shape} (Ημέρες, Νομίσματα)")

# 2. Υπολογισμός Λογαριθμικών Αποδόσεων (Log-Returns)
# Ο τύπος ln(P_t / P_{t-1}). Δημιουργεί NaN στην πρώτη γραμμή, οπότε κάνουμε dropna(how='all')
log_returns_yahoo = np.log(prices_yahoo / prices_yahoo.shift(1)).dropna(how='all')
log_returns_binance = np.log(prices_binance / prices_binance.shift(1)).dropna(how='all')
log_returns_coinbase = np.log(prices_coinbase / prices_coinbase.shift(1)).dropna(how='all')

# 3. Αποθήκευση στα τελικά CSVs
log_returns_yahoo.to_csv('crypto_log_returns_yahoo.csv')
log_returns_binance.to_csv('crypto_log_returns_binance.csv')
log_returns_coinbase.to_csv('crypto_log_returns_coinbase.csv')

print("\n--- ΕΠΙΤΥΧΙΑ ---")
print("Τα τελικά CSVs με τα Log-Returns αποθηκεύτηκαν!")
print("Είναι έτοιμα να τροφοδοτήσουν το Baseline (M2-M3) και τα ST-GNNs (M4-M5).")

# Δείγμα από το Coinbase για να δούμε τα NaNs να δουλεύουν
print("\nΔείγμα Coinbase Log-Returns (Παρατηρήστε τα NaNs στο BNB-USD):")
print(log_returns_coinbase[['BTC-USD', 'ETH-USD', 'BNB-USD']].tail())

-> Άντληση δεδομένων από Yahoo Finance...
-> Άντληση δεδομένων από Binance API...
-> Άντληση δεδομένων από Coinbase API...
   [i] Το νόμισμα TRX-USD δεν βρέθηκε στο Coinbase (Θα μπει NaN).

--- Σύνοψη Διαστάσεων (Close Prices) ---
Yahoo Finance : (2987, 10) (Ημέρες, Νομίσματα)
Binance       : (1000, 10) (Ημέρες, Νομίσματα)
Coinbase      : (350, 10) (Ημέρες, Νομίσματα)

--- ΕΠΙΤΥΧΙΑ ---
Τα τελικά CSVs με τα Log-Returns αποθηκεύτηκαν!
Είναι έτοιμα να τροφοδοτήσουν το Baseline (M2-M3) και τα ST-GNNs (M4-M5).

Δείγμα Coinbase Log-Returns (Παρατηρήστε τα NaNs στο BNB-USD):
             BTC-USD   ETH-USD   BNB-USD
Date                                    
2026-03-02  0.045520  0.044276  0.031026
2026-03-03 -0.007232 -0.022253 -0.006624
2026-03-04  0.061675  0.070454  0.038404
2026-03-05 -0.025016 -0.026007 -0.015679
2026-03-06 -0.038490 -0.044926 -0.029505
